## Laboratory 13 — cloud variant
### Exercise 1 — Cloud LLM inference and manual tool calling

In [ ]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.environ["GEMINI_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

chat_response = client.chat.completions.create(
    model="gemini-3.5-flash",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "How important is LLMOps on scale 0-10?"},
    ],
    max_completion_tokens=1000,
    # turn off thinking, analogous to Qwen's enable_thinking=False
    reasoning_effort="none",  # disable Gemini's extended thinking for this simple test
    # We turn reasoning off in this lab for speed.
)
content = chat_response.choices[0].message.content.strip()
print("Response:\n", content)

Response:
 On a scale of 0 to 10, the importance of LLMOps (Large Language Model Operations) is a **9 out of 10** overall, and a definitive **10 out of 10** for enterprise-grade, production-level applications.

Here is the breakdown of why it rates so highly, and why it isn’t a perfect 10 for *every* scenario:

### Why it is a 9/10 (The Reality)

*   **Going from Prototype to Production is Brutal (10/10):** Building a cool LLM demo in a Jupyter Notebook takes an afternoon. Deploying that same model to serve 100,000 users safely, quickly, and cheaply is incredibly difficult. LLMOps bridges this gap.
*   **The "Three Headaches" of LLMs:**
    1.  **Cost:** LLM APIs and hosting GPUs are incredibly expensive. LLMOps (through caching, prompt routing, and model quantization) is the only way to keep costs sustainable.
    2.  **Latency:** Users expect instant answers. LLMOps manages the infrastructure required to keep response times low.
    3.  **Hallucinations & Drift:** LLMs are non-determ

In [ ]:
import datetime
import json
import os
from typing import Callable

from dotenv import load_dotenv
from openai import OpenAI


def make_llm_request(prompt: str) -> str:
    load_dotenv()
    client = OpenAI(
        api_key=os.environ["GEMINI_API_KEY"],
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    messages = [
        {"role": "system", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model="gemini-3.5-flash",
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            reasoning_effort="low",
        )
        resp_message = response.choices[0].message
        messages.append(resp_message.model_dump(exclude_none=True))

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "get_current_date",
                "description": 'Get current date in the format "Year-Month-Day" (YYYY-MM-DD).',
                "parameters": {},
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_weather_forecast",
                "description": "Get weather forecast at given country, city, and date.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "country": {
                            "type": "string",
                            "description": "The country the city is in.",
                        },
                        "city": {
                            "type": "string",
                            "description": "The city to get the weather for.",
                        },
                        "date": {
                            "type": "string",
                            "description": (
                                "The date to get the weather for, "
                                'in the format "Year-Month-Day" (YYYY-MM-DD). '
                                "At most 4 weeks into the future."
                            ),
                        },
                    },
                    "required": ["country", "city", "date"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "get_current_date": current_date_tool,
        "get_weather_forecast": weather_forecast_tool,
    }

    return tool_definitions, tool_name_to_callable


def current_date_tool() -> str:
    return datetime.date.today().isoformat()


def weather_forecast_tool(country: str, city: str, date: str) -> str:
    if country.lower() in {"united kingdom", "uk", "england"}:
        return "Fog and rain"
    else:
        return "Sunshine"


if __name__ == "__main__":
    prompt = "What will be weather in Birmingham, UK in two weeks?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    print()

    prompt = "What will be weather in Warsaw the day after tomorrow?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

    print()

    prompt = "What will be weather in New York in two months?"
    response = make_llm_request(prompt)
    print("Response:\n", response)

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'vgsmvvi4', 'function': {'arguments': '{}', 'name': 'get_current_date'}, 'type': 'function', 'extra_content': {'google': {'thought_signature': 'EvYBCvMBAQw51sdMMBlP0X3Y8gUlKSXJ1TXXzUnudwy8JZSkehY97N3BrrgyPnMBTBg8LHTCloxF/73aeDte1rTQAaixxrN10zaeuyb9wCOQ88rtTj1RIwcFiM3mCEjRakNVUIterc8CgXKGrwUPi4ktFP+u/9GxK740m6CxQBJITDcRYvNmCUhWGBytNk0RHGG0twrCmZq/LgaKiV4qOnFwKdO72qdmJ2gH639fIjlMZHE9r4enZ0I3RH/LeLJ0PWWNfk5wK7AkZUA/scx/A2HBakqGJpGraBjTc+hAcaUjRfht2AgjTGlbUUhl+HyrKDx6/0H/d+ih'}}}]}

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'aaegnz65', 'function': {'arguments': '{"country":"UK","city":"Birmingham","date":"2026-07-10"}', 'name': 'get_weather_forecast'}, 'type': 'function', 'extra_content': {'google': {'thought_signature': 'Et

## Prompt: "What will be weather in Warsaw the day after tomorrow?"
### Logged intermediate messages and tool call sequence:

The model produced three messages in sequence:

Message 1 — tool call: `get_current_date`:
The model called `get_current_date` with no arguments.
Tool output: "2026-06-26" -> This value was appended to the conversation as a tool message.

Reasoning step (date computation by the model):
Using the returned date (2026-06-26), the model interpreted “the day after tomorrow” as +2 days and computed the target date:
2026-06-26 + 2 days = 2026-06-28
This computation was performed by the model (not by any tool).

Message 2 — tool call: `get_weather_forecast`:
The model called get_weather_forecast with: city -> Warsaw, country -> Poland, date -> 2026-06-28
Tool output: "Sunshine"

Message 3 — final response (no tool calls):
The model generated the final answer using the tool result.

Summary of the sequence:
1. `get_current_date()` → 2026-06-26
2. Model computes: “day after tomorrow” → 2026-06-28
3. `get_weather_forecast(Warsaw, Poland, 2026-06-28)` → "Sunshine"
4. Final answer returned to the user

## Prompt: "What will be weather in New York in two months?"
### How the model handles the constraint ("at most 4 weeks into the future") violation:

The model first called `get_current_date`, which returned 2026-06-26. It then determined that the requested forecast date would fall in late August 2026 (approximately two months ahead). Based on the tool schema, the model recognized that this exceeded the supported forecasting range of "at most four weeks into the future".

Instead of calling `get_weather_forecast` with an unsupported date, the model stopped and responded directly to the user. It explained that the requested forecast was outside the tool's supported range and suggested appropriate alternatives.

The model correctly adhered to the constraint specified in the tool description "at most 4 weeks into the future". It neither attempted to call `get_weather_forecast` with an out-of-range date nor fabricated a weather forecast. Instead, it transparently communicated the limitation and offered a helpful alternative response.